<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_1_k_windows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3_2_build_X_y_and_scaling


## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Generación de ventanas X e y para train, valid y test

    En este paso se generan las ventanas de entrada (X) y los targets (y) para los conjuntos de entrenamiento, validación y prueba.
    Se trabaja con un window_size de 90 minutos y se construyen datasets independientes para cada horizonte de predicción: 30, 60 y 90 minutos.

3. Escalado de ventanas

    En este paso se aplica un proceso de normalización/estandarización a las ventanas generadas, utilizando un scaler entrenado únicamente con el set de entrenamiento para cada horizonte de predicción.
    De esta forma, se aseguran valores comparables entre features y se evita data leakage.
    El scaler ajustado se guarda para poder transformar consistentemente los conjuntos de validación y prueba.

4. Guardado de ventanas escaladas

    En este paso se almacenan en disco las ventanas ya escaladas de train, valid y test, correspondientes a cada horizonte de predicción (30, 60 y 90 minutos).
    Esto permite reutilizar los datasets en etapas posteriores sin necesidad de repetir el preprocesamiento.



## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [2]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [3]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [4]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [5]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib

## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [27]:
def load_data(fold: str, data: str):

    data_path = f'{drive_path}/5_transformer_90_model/5_0_k_folds/fold_{fold}/{data}_{fold}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar orden cronológico por índice
    df = df.sort_index()

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [28]:
k_folds = [1, 2, 3, 4, 5]

In [40]:
for k in k_folds:
  print(f'Cargando datos de Fold {k}..')
  globals()[f'mnq_train_{k}'] = load_data(str(k), 'train')
  globals()[f'mnq_valid_{k}'] = load_data(str(k), 'valid')
  globals()[f'mnq_test_{k}'] = load_data(str(k), 'test')

Cargando datos de Fold 1..
Cargando datos de Fold 2..
Cargando datos de Fold 3..
Cargando datos de Fold 4..
Cargando datos de Fold 5..


### 1.2. Información de datasets


In [60]:
def info_dataset(df, name: str):

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  #print(f"\t{name}:\t{num_dias} días")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  #print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  print(f"\t{name}:\t{num_dias} días con {int(promedio_por_fecha)} registros")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo

  #print(f"\tHora diaria de inicio {primer_hora}")
  #print(f"\tHora diaria de final {ultima_hora}")
  #print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [61]:
for k in k_folds:
  print(f'Fold {k}:')
  info_dataset(globals()[f'mnq_train_{k}'],f'mnq_train_{k}')
  info_dataset(globals()[f'mnq_valid_{k}'],f'mnq_valid_{k}')
  info_dataset(globals()[f'mnq_test_{k}'],f'mnq_test_{k}')
  print ('\n')

Fold 1:
	mnq_train_1:	589 días con 301 registros
	mnq_valid_1:	118 días con 301 registros
	mnq_test_1:	132 días con 301 registros


Fold 2:
	mnq_train_2:	707 días con 301 registros
	mnq_valid_2:	118 días con 301 registros
	mnq_test_2:	132 días con 301 registros


Fold 3:
	mnq_train_3:	825 días con 301 registros
	mnq_valid_3:	118 días con 301 registros
	mnq_test_3:	132 días con 301 registros


Fold 4:
	mnq_train_4:	943 días con 301 registros
	mnq_valid_4:	118 días con 301 registros
	mnq_test_4:	132 días con 301 registros


Fold 5:
	mnq_train_5:	1061 días con 301 registros
	mnq_valid_5:	118 días con 301 registros
	mnq_test_5:	132 días con 301 registros




## 2. Generación de ventanas X e y para train, valid y test

Definimos el target de cada horizonte:

In [65]:
target_col_90 = "target_return_90"

Luego definimos el listado de features para cada horizonte:

In [69]:
features_90 = features_90 = [
    col for col in mnq_train_1.columns
    if col not in ["date", "target_return_90"]
]

In [70]:
features_90

['open',
 'high',
 'low',
 'close',
 'volume',
 'ire_60',
 'rev_mom_z_90',
 'roc_60',
 'bb_60',
 'momentum_5',
 'roc_20',
 'rev_mom_vol_z_60']

El `window_size` está condicionado por el feature que más historial necesita, en nuestro caso `roc_90`, `rev_mom_z_90` y `ire_90` necesitan 90 minutos previos para poder calcular su primer valor válido.

Si hacemos más corto el `window_size` corremos el riesgo de perder información o generar NaNs.

Y un `window_size` más largo?  por ahora experimentemos con 90.


In [71]:
window_size = 90

Definimos las rutas para las ventanas:

In [73]:
# Ruta base donde guardarás los folds
ruta_k_windows = f'{drive_path}/5_transformer_90_model/5_1_k_windows'
os.makedirs(ruta_k_windows, exist_ok=True)

In [74]:
def xy_paths_for_fold(k: int):
    # Crear carpeta del fold
    fold_path = os.path.join(ruta_k_windows, f"fold_{k}")
    os.makedirs(fold_path, exist_ok=True)
    base = f'{drive_path}/5_transformer_90_model/5_1_k_windows/fold_{k}'
    return {
        "X_train": f"{base}/X_train_{k}.npz",
        "y_train": f"{base}/y_train_{k}.npz",
        "X_valid": f"{base}/X_valid_{k}.npz",
        "y_valid": f"{base}/y_valid_{k}.npz",
        "X_test":  f"{base}/X_test_{k}.npz",
        "y_test":  f"{base}/y_test_{k}.npz",
    }

In [75]:
K = 5

rutas_ventanas = {
    k: xy_paths_for_fold(k)
    for k in range(1, K + 1)
}

In [76]:
rutas_ventanas

{1: {'X_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_train_1.npz',
  'y_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_train_1.npz',
  'X_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_valid_1.npz',
  'y_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_valid_1.npz',
  'X_test': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_test_1.npz',
  'y_test': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_test_1.npz'},
 2: {'X_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_train_2.npz',
  'y_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/y_train_2.npz',
  'X_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_valid_2.npz'

### 2.0. Funciones

#### Función para generar ventanas

Genera ventana consecutivas y no aleatorias, es decir: ventanas deslizantes (sliding windows) dentro de cada día.

- Agrupamiento diario: Cada iteración toma un día completo del dataset (917 días en total). Dentro de ese grupo tenemos 301 registros minuto a minuto.

- Iteración dentro del día: genera una ventana que empieza en el minuto i  termina en i + windows_size-1.

Ejemplo si window_size = 90 y tenemos 301 minutos:

  | Iteración | Ventana usada     | Target extraído       |
  | --------- | ----------------- | --------------------- |
  | i = 0     | registros 0–89    | target = registro 89  |
  | i = 1     | registros 1–90    | target = registro 90  |
  | i = 2     | registros 2–91    | target = registro 91  |
  | ...       | ...               | ...                   |
  | i = 210   | registros 210–299 | target = registro 299 |

Esto da 301 - 90 = 211 ventanas por día, todas consecutivas.


In [77]:
#Función para generar ventanas y vectorizarlas
def generar_ventanas(df, features, target_col, window_size):
    X, y = [], []

    #1. Agrupamiento diario: Cada iteración toma un día completo del dataset (917 días en total). Dentro de ese grupo tenemos 301 registros minuto a minuto.
    for fecha, grupo in tqdm(df.groupby("date"), desc="Procesando días"):
        grupo = grupo.reset_index(drop=True)

        #2. Iteración dentro del día: genera una ventana que empieza en el minuto i  termina en i + windows_size-1.
        for i in range(len(grupo) - window_size):
            ventana = grupo.loc[i:i+window_size-1, features]
            if ventana.isnull().any().any():
                continue

            # 3. Toma las columnas listadas en features (por ejemplo, 10 features por minuto) y las aplanas en un solo vector 1D de longitud window_size × len(features) (= 900 si window_size=90 y len(features)=10).
            vector = ventana.values.flatten()

            #4. El target de la ventana es el valor del registro en el último minuto de la ventana, o sea el del minuto i + window_size - 1.
            #(No es “futuro”, sino el último dentro de la ventana).
            target = grupo.loc[i+window_size-1, target_col]

            X.append(vector)
            y.append(target)

      # Se obtiene:
      # X.shape = (n_ventanas_totales, window_size * n_features)
      # y.shape = (n_ventanas_totales,)
    return np.array(X), np.array(y)

Cada ventana contiene:

- 90 minutos consecutivos de datos de un mismo día.
- En cada minuto, 10 o 12 features (por ejemplo: open, high, low, close, volume, etc.).
- Esos 90×10 (ó 12) valores se aplanan en un vector de 900 (ó 1080) elementos.
- El target asociado es el valor del minuto siguiente al final de la ventana (o del último minuto, según definas).
- Por día se generan 301 − 90 = 211 ventanas, todas superpuestas y consecutivas.
- Repetido en los 917 días de train,  se obtiene 193 487 ventanas en total.



#### Función para generar xy de acuerdo a horizonte de tiempo

In [78]:
def generar_xy (
    df_train,
    df_valid,
    df_test,
    features,
    target: str,
    window_size: int,
    path_xy_train : str,
    path_xy_valid : str,
    path_xy_test : str
    ):

  if not os.path.exists(path_xy_train):
      print(f'El archivo no existe -> Generando X_train e y_train para {target}: ')
      X_train, y_train = generar_ventanas(df_train, features, target, window_size)
      np.savez_compressed(path_xy_train, X=X_train, y=y_train)
      print("Guardado:", path_xy_train)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_train)
      data_train = np.load(path_xy_train)
      X_train, y_train = data_train["X"], data_train["y"]

  if not os.path.exists(path_xy_valid):
      print('El archivo no existe -> Generando X_valid e y_valid: ')
      X_valid, y_valid = generar_ventanas(df_valid, features, target, window_size)
      np.savez_compressed(path_xy_valid, X=X_valid, y=y_valid)
      print("Guardado:", path_xy_valid)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_valid)
      data_valid = np.load(path_xy_valid)
      X_valid, y_valid = data_valid["X"], data_valid["y"]

  if not os.path.exists(path_xy_test):
      print('El archivo no existe -> Generando X_test e y_test: ')
      X_test, y_test = generar_ventanas(df_test, features, target, window_size)
      np.savez_compressed(path_xy_test, X=X_test, y=y_test)
      print("Guardado:", path_xy_test)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_test)
      data_test = np.load(path_xy_test)
      X_test, y_test = data_test["X"], data_test["y"]

  return X_train, y_train, X_valid, y_valid, X_test, y_test

#### Función para revisar información de ventanas

In [79]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

### 2.1. Para horizonte de 30 minutos (`target_return_30`)

In [ ]:
for k in k_folds:
  print(f'Fold {k}:')
  X_train, y_train, X_valid, y_valid, X_test, y_test = generar_xy(
      globals()[f'mnq_train_{k}'],
      globals()[f'mnq_valid_{k}'],
      globals()[f'mnq_test_{k}'],
      features_90,
      target_col_90,
      window_size,
      rutas_ventanas[k]['X_train'],
      rutas_ventanas[k]['X_valid'],
      rutas_ventanas[k]['X_test']
      )

Fold 1:
El archivo no existe -> Generando X_train e y_train para target_return_90: 


Procesando días: 100%|██████████| 589/589 [02:08<00:00,  4.60it/s]


Guardado: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_train_1.npz
El archivo no existe -> Generando X_valid e y_valid: 


Procesando días: 100%|██████████| 118/118 [00:24<00:00,  4.88it/s]


Guardado: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_valid_1.npz
El archivo no existe -> Generando X_test e y_test: 


Procesando días: 100%|██████████| 132/132 [00:27<00:00,  4.81it/s]


Guardado: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_test_1.npz
Fold 2:
El archivo no existe -> Generando X_train e y_train para target_return_90: 


Procesando días:  64%|██████▎   | 449/707 [01:33<00:53,  4.81it/s]

In [ ]:
xy_info( '90', X_train, y_train, X_valid, y_valid, X_test, y_test)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000129, std=0.004762, min=-0.040748, max=0.051227

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000090, std=0.005930, min=-0.029164, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000138, std=0.004433, min=-0.030108, max=0.026213


## 3. Escalado de ventanas

En este punto se escalan las ventanas de entrada para que todas las features tengan la misma magnitud, entrenando el scaler con los datos de entrenamiento y aplicándolo luego a validación y test.

### 3.0. Funciones


#### Función para elegir escalador o cargar escalador

In [ ]:
def _choose_scaler(scaler_type="standard"):
    st = scaler_type.lower()
    if st in ["standard", "z", "zscore"]:
        return StandardScaler()
    elif st in ["minmax", "min_max"]:
        return MinMaxScaler()
    else:
        raise ValueError("scaler_type debe ser 'standard' o 'minmax'")

#### Función para entrenar un scaler en datos secuenciales 3D (ventanas), tratándolos como una sola tabla 2D de features.

In [ ]:
# -------------------------------------------------------------------------
# _fit_on_3d
#
# Entrada: X_train_3d con forma (n, W, F)
#   n = número de muestras (ventanas)
#   W = lookback (número de pasos en cada ventana)
#   F = número de features por paso
#
# Qué hace:
# - Aplana las dos primeras dimensiones (n, W) → queda una matriz de (n*W, F).
# - Convierte todas las secuencias en un dataset tabular de features.
# - Ajusta el scaler (ej. StandardScaler) sobre todos los valores de todas
#   las ventanas y pasos, feature por feature.
#
# Resultado: devuelve un scaler entrenado con la estadística global de cada
# feature (media, std, min, max, según el tipo de scaler utilizado).
# -------------------------------------------------------------------------

def _fit_on_3d(X_train_3d, scaler):
    n, W, F = X_train_3d.shape
    scaler.fit(X_train_3d.reshape(-1, F))
    return scaler

#### Función para aplicar el scaler de _fit_on_3d y devolver los datos escalados, manteniendo la estructura original (n, W, F).

In [ ]:
# -------------------------------------------------------------------------
# _transform_3d
#
# Entrada: X_3d con forma (n, W, F)
#   n = número de muestras (ventanas)
#   W = lookback (número de pasos en cada ventana)
#   F = número de features por paso
#
# Qué hace:
# - Aplana las dos primeras dimensiones (n, W) → queda una matriz de (n*W, F).
# - Aplica la transformación del scaler entrenado (ej. StandardScaler).
# - Restaura la forma original (n, W, F) para conservar la estructura 3D
#   necesaria en modelos secuenciales (RNN, LSTM, Transformers).
#
# Resultado: devuelve el mismo dataset 3D pero con todos los features escalados
# de manera consistente en cada ventana y paso de tiempo.
# -------------------------------------------------------------------------

def _transform_3d(X_3d, scaler):
    n, W, F = X_3d.shape
    Xf = X_3d.reshape(-1, F)
    Xs = scaler.transform(Xf).reshape(n, W, F)
    return Xs

#### Función para escalar y guardar escalador

In [ ]:
import os
import joblib
import numpy as np

def scale_and_save(
    X_train,
    X_valid=None,
    X_test=None,
    scaler_type="standard",
    scaler_path=f"{drive_path}/3_dataset_preparation/global_scaler.pkl",
    window_size=None,   # si X_* están en 2D (n_samples, window_size*len(features_{target})), pasá window_size para escalar por feature
    verbose=True,
    reuse_if_exists=True,  # <-- NUEVO: si True y existe scaler_path, lo reutiliza
):
    """
    Escala X_train (y opcionalmente valid/test) y guarda el escalador.
    - Si existe un scaler en `scaler_path` y `reuse_if_exists=True`, lo carga y NO vuelve a hacer fit.
    - Si no existe, crea uno nuevo, hace fit con X_train y lo guarda en `scaler_path`.

    - Si X_* es 3D: (n, W, F) -> fit por feature.
    - Si X_* es 2D: (n, W*F). Si pasás window_size=W, reescala por feature reconstruyendo 3D; si no, escala columnas tal cual.

    Return:
        X_train_scaled, X_valid_scaled (o None), X_test_scaled (o None), scaler
    """

    # ---------------------------------------------------------
    # 1) Obtener scaler: cargar si existe, o crear y ajustar si no
    # ---------------------------------------------------------
    fitted = False

    if reuse_if_exists and os.path.exists(scaler_path):
        scaler = joblib.load(scaler_path)
        fitted = True  # ya viene entrenado
        if verbose:
            print(f"🔁 Usando scaler existente de: {scaler_path}")
    else:
        scaler = _choose_scaler(scaler_type)
        if verbose:
            if os.path.exists(scaler_path):
                print(f"scaler_path existe pero reuse_if_exists=False → se creará y ajustará un nuevo scaler.")
            else:
                print("Scaler no encontrado → se creará y ajustará uno nuevo.")

    # ---------------------------------------------------------
    # 2) Escalado según dimensión de X_train
    # ---------------------------------------------------------
    if X_train.ndim == 3:
        # X_train: (n, W, F)
        if not fitted:
            scaler = _fit_on_3d(X_train, scaler)

        X_train_s = _transform_3d(X_train, scaler)
        X_valid_s = _transform_3d(X_valid, scaler) if X_valid is not None else None
        X_test_s  = _transform_3d(X_test,  scaler) if X_test  is not None else None

    elif X_train.ndim == 2:
        n, tot = X_train.shape
        if window_size is not None:
            # Reescalar por feature: reconstruyo 3D -> escalo -> vuelvo a 2D
            assert tot % window_size == 0, "total de columnas no divisible por window_size"
            F = tot // window_size

            def to3d(X2d):
                return X2d.reshape(X2d.shape[0], window_size, F)

            Xtr3 = to3d(X_train)
            if not fitted:
                scaler = _fit_on_3d(Xtr3, scaler)

            X_train_s = _transform_3d(Xtr3, scaler).reshape(n, tot)

            if X_valid is not None:
                Xva3 = to3d(X_valid)
                X_valid_s = _transform_3d(Xva3, scaler).reshape(X_valid.shape[0], tot)
            else:
                X_valid_s = None

            if X_test is not None:
                Xte3 = to3d(X_test)
                X_test_s = _transform_3d(Xte3, scaler).reshape(X_test.shape[0], tot)
            else:
                X_test_s = None
        else:
            # Escalado columna a columna (no reconstruye 3D)
            if not fitted:
                scaler.fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_valid_s = scaler.transform(X_valid) if X_valid is not None else None
            X_test_s  = scaler.transform(X_test)  if X_test  is not None else None
    else:
        raise ValueError("X_train debe ser 2D o 3D.")

    # ---------------------------------------------------------
    # 3) Guardar escalador (el que efectivamente se usó)
    # ---------------------------------------------------------
    joblib.dump(scaler, scaler_path)
    if verbose:
        print(f"✅ Scaler usado/actualizado guardado en: {scaler_path}")
        print("Shapes escaladas:",
              "X_train", X_train_s.shape,
              "| X_valid", None if X_valid is None else X_valid_s.shape,
              "| X_test",  None if X_test  is None  else X_test_s.shape)

    return X_train_s, X_valid_s, X_test_s, scaler


### 3.2. Verificación de existencia de ventanas escaladas:

Definimos las rutas

In [ ]:
def xy_scaled_path (target: str):
  path_xy_train = f"{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz"
  path_xy_valid = f"{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz"
  path_xy_test =  f"{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz"
  return path_xy_train, path_xy_valid, path_xy_test

Para las ventanas con el dataset completo:

In [ ]:
path_xy_train_30_scaled, path_xy_valid_30_scaled, path_xy_test_30_scaled = xy_scaled_path('30')


Para las ventanas con el 20% del dataset:

In [ ]:
path_xy_train_30_ss_scaled, path_xy_valid_30_ss_scaled, path_xy_test_30_ss_scaled = xy_scaled_path('30_ss_20')


Función para guardar o cargar ventanas escaladas:

In [ ]:
def save_or_load_scaled_data(path_train, path_valid, path_test, horizonte, X_train, X_valid, X_test, y_train, y_valid, y_test):

  paths = {
    "train": path_train,
    "valid": path_valid,
    "test":  path_test
    }

  scaler_path = f"{drive_path}/3_dataset_preparation/global_scaler_{horizonte}.pkl"

  # Verificar existencia conjunta
  if all(os.path.exists(p) for p in paths.values()):
      print("Los tres archivos existen → Cargando ventanas escaladas")

      data_train_s = np.load(path_train)
      X_train_s, y_train = data_train_s["X"], data_train_s["y"]

      data_valid_s = np.load(path_valid)
      X_valid_s, y_valid = data_valid_s["X"], data_valid_s["y"]

      data_test_s = np.load(path_test)
      X_test_s, y_test = data_test_s["X"], data_test_s["y"]

      print("Carga completa.")


  else:
      print("Alguno de los archivos no existe → Generando ventanas escaladas")

      if os.path.exists(scaler_path):
        print(f"Scaler existente → cargando global_scaler_{horizonte}.pkl")
        scaler = joblib.load(scaler_path)
      else:
        print(f"Scaler no existe → creando global_scaler_{horizonte}.pkl")

      X_train_s, X_valid_s, X_test_s, scaler = scale_and_save(
        X_train,
        X_valid,
        X_test,
        scaler_type="standard",
        scaler_path=scaler_path,
        window_size=90,        # ← IMPORTANTE para su caso
        verbose=True,
        reuse_if_exists=True   # ← usa scaler guardado si existe
    )


      np.savez_compressed(path_train, X=X_train_s, y=y_train)
      np.savez_compressed(path_valid, X=X_valid_s, y=y_valid)
      np.savez_compressed(path_test,  X=X_test_s,  y=y_test)

      print("Guardado completo.")

  return X_train_s, X_valid_s, X_test_s

### 3.1. Escalado para horizonte de 30 minutos (`target_return_30`)

In [ ]:
X_train_30_s, X_valid_30_s, X_test_30_s = save_or_load_scaled_data(
    path_xy_train_30_scaled, path_xy_valid_30_scaled, path_xy_test_30_scaled, '30', X_train_30, X_valid_30, X_test_30, y_train_30,  y_valid_30,  y_test_30 )

Los tres archivos existen → Cargando ventanas escaladas
Carga completa.


Ventanas subseteadas al 10%

In [ ]:
X_train_30_ss_s, X_valid_30_ss_s, X_test_30_ss_s = save_or_load_scaled_data(
    path_xy_train_30_ss_scaled, path_xy_valid_30_ss_scaled, path_xy_test_30_ss_scaled, '30', X_train_30_ss, X_valid_30_ss, X_test_30_ss,  y_train_30_ss,  y_valid_30_ss,  y_test_30_ss)

Alguno de los archivos no existe → Generando ventanas escaladas
Scaler existente → cargando global_scaler_30.pkl
🔁 Usando scaler existente de: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_30.pkl
✅ Scaler usado/actualizado guardado en: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_30.pkl
Shapes escaladas: X_train (38613, 900) | X_valid (8229, 900) | X_test (8229, 900)
Guardado completo.


### 3.2. Escalado para horizonte de 60 minutos (`target_return_60`)

In [ ]:
X_train_60_s, X_valid_60_s, X_test_60_s = save_or_load_scaled_data(
    path_xy_train_60_scaled, path_xy_valid_60_scaled, path_xy_test_60_scaled, '60', X_train_60, X_valid_60, X_test_60, y_train_60,  y_valid_60,  y_test_60 )

Los tres archivos existen → Cargando ventanas escaladas
Carga completa.


In [ ]:
X_train_60_ss_s, X_valid_60_ss_s, X_test_60_ss_s = save_or_load_scaled_data(
    path_xy_train_60_ss_scaled, path_xy_valid_60_ss_scaled, path_xy_test_60_ss_scaled, '60', X_train_60_ss, X_valid_60_ss, X_test_60_ss,  y_train_60_ss,  y_valid_60_ss,  y_test_60_ss)

Alguno de los archivos no existe → Generando ventanas escaladas
Scaler existente → cargando global_scaler_60.pkl
🔁 Usando scaler existente de: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_60.pkl
✅ Scaler usado/actualizado guardado en: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_60.pkl
Shapes escaladas: X_train (38613, 1080) | X_valid (8229, 1080) | X_test (8229, 1080)
Guardado completo.


### 3.3. Escalado para horizonte de 90 minutos (`target_return_90`)

In [ ]:
X_train_90_s, X_valid_90_s, X_test_90_s = save_or_load_scaled_data(
    path_xy_train_90_scaled, path_xy_valid_90_scaled, path_xy_test_90_scaled, '90', X_train_90, X_valid_90, X_test_90, y_train_90,  y_valid_90,  y_test_90 )

Los tres archivos existen → Cargando ventanas escaladas
Carga completa.


In [ ]:
X_train_90_ss_s, X_valid_90_ss_s, X_test_90_ss_s = save_or_load_scaled_data(
    path_xy_train_90_ss_scaled, path_xy_valid_90_ss_scaled, path_xy_test_90_ss_scaled, '90', X_train_90_ss, X_valid_90_ss, X_test_90_ss,  y_train_90_ss,  y_valid_90_ss,  y_test_90_ss)

Alguno de los archivos no existe → Generando ventanas escaladas
Scaler existente → cargando global_scaler_90.pkl
🔁 Usando scaler existente de: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_90.pkl
✅ Scaler usado/actualizado guardado en: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_90.pkl
Shapes escaladas: X_train (38613, 1080) | X_valid (8229, 1080) | X_test (8229, 1080)
Guardado completo.
